# combine the plots of the 11 methods in one figure

In [1]:
import os
from PIL import Image
import pathlib 
from pathlib import Path
DPI = 600                        # high-res for print publications
TEXTWIDTH_INCHES = 6.5           # typical \textwidth for A4 with 1-inch margins
TEXTWIDTH_PX = int(TEXTWIDTH_INCHES * DPI)   # 3900 px
CELL_PAD = 20                    # gap between sub-images (px at 600 DPI)
COLS = 3                         # figures per row


def combine_images_grid(image_paths, output_path, cols=COLS):
    """Arrange images in a grid (cols per row), scaled to fit LaTeX \\textwidth.

    - Each sub-image is individually resized with LANCZOS (high-quality anti-aliased).
    - If a source image is smaller than the target cell, it is up-scaled with
      BICUBIC to avoid blocky artefacts from nearest-neighbour interpolation.
    - Output is saved as lossless PNG at ``DPI`` resolution.
    """
    images = [Image.open(p).convert("RGB") for p in image_paths]
    n = len(images)
    rows = (n + cols - 1) // cols

    # Cell dimensions that tile exactly into TEXTWIDTH_PX
    cell_w = (TEXTWIDTH_PX - (cols - 1) * CELL_PAD) // cols

    # Preserve aspect ratio based on median image dimensions (more robust than first-only)
    aspects = [img.size[1] / img.size[0] for img in images]
    median_aspect = sorted(aspects)[len(aspects) // 2]
    cell_h = int(cell_w * median_aspect)

    # Choose resampling filter per image: LANCZOS for downscale, BICUBIC for upscale
    resized = []
    for img in images:
        filt = Image.LANCZOS if img.size[0] >= cell_w else Image.BICUBIC
        resized.append(img.resize((cell_w, cell_h), filt))

    total_w = cols * cell_w + (cols - 1) * CELL_PAD
    total_h = rows * cell_h + (rows - 1) * CELL_PAD

    canvas = Image.new("RGB", (total_w, total_h), "white")
    for i, img in enumerate(resized):
        x = (i % cols) * (cell_w + CELL_PAD)
        y = (i // cols) * (cell_h + CELL_PAD)
        canvas.paste(img, (x, y))

    canvas.save(output_path, dpi=(DPI, DPI), compress_level=1)  # fast, lossless PNG


# ===== CONFIG =====
base_dir = "/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs"
methods = [
    "bbknn_full_hvg_seed0", "combat_full_hvg_seed0", "fastmnn_full_hvg_seed0", "harmony_full_hvg_seed0",
    "liger_full_hvg_seed0", "mnn_full_hvg_seed0", "scanorama_full_hvg_seed0", "scanvi_full_hvg_seed0",
    "scgen_full_hvg_seed0", "scvi_full_hvg_seed0", "seurat_full_hvg_seed0"
]

plot_types = [
    "umap_compartment.png",  
    "umap_donor_id.png",
    "umap_platform.png",
    "umap_technology.png",

]

# For each plot type, collect the corresponding plot from each method's plot_pub folder
for plot in plot_types:
    paths = [
        os.path.join(base_dir, m, "plots_pub", plot)
        for m in methods
        if os.path.isfile(os.path.join(base_dir, m, "plots_pub", plot))
    ]
    if not paths:
        print(f"⚠ No figures found for {plot}, skipping.")
        continue

    output = Path("/data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/combined_plots") / f"combined_{plot}"
    os.makedirs(output.parent, exist_ok=True)
    combine_images_grid(paths, output, cols=COLS)
    print(f"✓ Saved {output}")

✓ Saved /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/combined_plots/combined_umap_compartment.png
✓ Saved /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/combined_plots/combined_umap_donor_id.png
✓ Saved /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/combined_plots/combined_umap_platform.png
✓ Saved /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/combined_plots/combined_umap_technology.png


# construct all the scores in unified table

In [1]:
"""
Iterates over method output folders in:
Results/Integration/full_hvg_seed0/runs

Aggregates:
- Selected metrics from metrics.json
- Top-level config parameters from config.json
- Performance data from perf_log.csv

Outputs:
1. Methods × Metrics table
2. Methods × Performance logs table
3. Config parameters table
4. Complete combined table
"""

import json
import re
from pathlib import Path
from typing import Dict, Any
import pandas as pd
import numpy as np
from IPython.display import display

# =============================================================================
# PATHS
# =============================================================================
RUNS_DIR = Path("/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs")
OUT_DIR = Path("/data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_DIR = OUT_DIR / "metrics"
PERF_DIR = OUT_DIR / "performance"
CONFIG_DIR = OUT_DIR / "configs"

for d in [METRICS_DIR, PERF_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# METHODS FROM SCREENSHOT
# =============================================================================
METHODS = [
    "bbknn",
    "combat",
    "fastmnn",
    "harmony",
    "liger",
    "scanorama",
    "scanvi",
    "scgen",
    "scvi",
    "seurat",
]

# Folder names exactly as shown in screenshot
METHOD_TO_FOLDER = {
    method: f"{method}_full_hvg_seed0"
    for method in METHODS
}

# =============================================================================
# CURATED METRICS TO KEEP
# =============================================================================
CURATED_METRICS = [
    "NMI",
    "ARI",
    "cell_type_ASW",
    "isolated_label_F1",
    "isolated_label_ASW",
    "cell_cycle_conservation",
    "hvg_conservation",
    "trajectory_conservation",
    "trajectory_conservation_note",
    "batch_ASW",
    "pcr_comparison",
    "kBET",
    "graph_connectivity",
    "iLISI",
    "cLISI",
]

# =============================================================================
# HELPERS
# =============================================================================
def load_json_allow_nan(json_path: Path) -> Dict[str, Any]:
    """
    Load JSON while replacing bare NaN with null so json.loads can parse it.
    """
    raw_text = json_path.read_text()
    raw_text = re.sub(r':\s*NaN', ': null', raw_text)
    return json.loads(raw_text)


def keep_top_level_scalars(d: Dict[str, Any], prefix: str = "") -> Dict[str, Any]:
    """
    Keep only top-level scalar values from a dictionary.
    Skips nested dicts/lists to avoid flattening logic.
    """
    out = {}
    for k, v in d.items():
        col = f"{prefix}{k}" if prefix else k
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[col] = v
    return out


def parse_perf_log(perf_path: Path) -> Dict[str, Any]:
    """
    Parse performance log and extract summary statistics.

    Expected schema:
    step, event (start|end), time_s, rss_gb, gpu_mem_gb
    """
    perf_data = {}

    try:
        perf_df = pd.read_csv(perf_path)

        end_rows = perf_df[perf_df["event"] == "end"] if "event" in perf_df.columns else perf_df

        if "time_s" in perf_df.columns:
            perf_data["perf_total_time_s"] = end_rows["time_s"].dropna().sum()

        if "rss_gb" in perf_df.columns:
            perf_data["perf_peak_rss_gb"] = perf_df["rss_gb"].dropna().max()

        if "gpu_mem_gb" in perf_df.columns:
            perf_data["perf_peak_gpu_gb"] = perf_df["gpu_mem_gb"].dropna().max()

        if "step" in perf_df.columns:
            perf_data["perf_num_steps"] = perf_df["step"].nunique()

    except Exception as e:
        print(f"Warning: Could not parse perf log {perf_path}: {e}")

    return perf_data


def parse_bool(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in ("true", "1", "yes", "y", "scaled"):
        return True
    if s in ("false", "0", "no", "n", "unscaled"):
        return False
    return np.nan


# =============================================================================
# SCAN RUN FOLDERS
# =============================================================================
print(f"Scanning runs directory: {RUNS_DIR}")
print(f"Expected methods: {METHODS}\n")

rows = []

for method in METHODS:
    run_folder = RUNS_DIR / METHOD_TO_FOLDER[method]

    if not run_folder.exists():
        print(f"Warning: folder not found for method '{method}': {run_folder}")
        continue

    metrics_path = run_folder / "metrics.json"
    config_path = run_folder / "config.json"
    perf_path = run_folder / "perf_log.csv"

    if not metrics_path.exists():
        print(f"Warning: metrics.json not found for method '{method}' in {run_folder}")
        continue

    row_data = {
        "method": method,
        "_run_name": run_folder.name,
        "_run_path": str(run_folder.relative_to(RUNS_DIR.parent.parent.parent)) if run_folder.exists() else str(run_folder),
    }

    # -------------------------------------------------------------------------
    # Load curated metrics only
    # -------------------------------------------------------------------------
    try:
        metrics = load_json_allow_nan(metrics_path)

        for metric in CURATED_METRICS:
            row_data[metric] = metrics.get(metric, np.nan)

    except Exception as e:
        print(f"ERROR reading {metrics_path}: {e}")
        continue

    # -------------------------------------------------------------------------
    # Load top-level scalar config values only
    # -------------------------------------------------------------------------
    if config_path.exists():
        try:
            config = load_json_allow_nan(config_path)
            config_scalar = keep_top_level_scalars(config, prefix="cfg_")
            row_data.update(config_scalar)

            # Bring a few useful config fields into main columns if present
            for key in ("dataset", "dataset_name"):
                if key in config:
                    row_data["dataset"] = config[key]

            for key in ("output", "output_type"):
                if key in config:
                    row_data["output_type"] = config[key]

            for key in ("scaled", "do_scale"):
                if key in config:
                    row_data["scaled"] = config[key]

            for key in ("batch_key", "label_key", "seed"):
                if key in config:
                    row_data[key] = config[key]

        except Exception as e:
            print(f"Warning: couldn't parse {config_path}: {e}")

    # -------------------------------------------------------------------------
    # Parse performance log
    # -------------------------------------------------------------------------
    if perf_path.exists():
        perf_data = parse_perf_log(perf_path)
        row_data.update(perf_data)

    rows.append(row_data)

print(f"\nSuccessfully processed {len(rows)} methods\n")

if not rows:
    raise ValueError("No runs were processed. Check RUNS_DIR and folder names.")

# =============================================================================
# BUILD DATAFRAME
# =============================================================================
df = pd.DataFrame(rows)

for col in ["method", "dataset", "output_type", "scaled"]:
    if col not in df.columns:
        df[col] = np.nan

if "scaled" in df.columns:
    df["scaled"] = df["scaled"].apply(parse_bool)

df = df.set_index("method")

# =============================================================================
# TABLE 1: METHODS × METRICS
# =============================================================================
print("=" * 60)
print("CREATING TABLE 1: Methods × Metrics")
print("=" * 60)

available_metrics = [m for m in CURATED_METRICS if m in df.columns]

print(f"\nFound {len(available_metrics)}/{len(CURATED_METRICS)} curated metrics:")
for i, metric in enumerate(available_metrics, 1):
    print(f"  {i:2d}. {metric}")

missing_metrics = set(CURATED_METRICS) - set(available_metrics)
if missing_metrics:
    print(f"\nMissing metrics: {sorted(missing_metrics)}")

if available_metrics:
    metrics_df = df[available_metrics]

    metrics_csv = METRICS_DIR / "methods_x_metrics.csv"
    metrics_excel = METRICS_DIR / "methods_x_metrics.xlsx"

    metrics_df.to_csv(metrics_csv)
    print(f"\n✓ Saved CSV:   {metrics_csv}")

    try:
        metrics_df.to_excel(metrics_excel, engine="openpyxl")
        print(f"✓ Saved Excel: {metrics_excel}")
    except Exception as e:
        print(f"Note: Could not save Excel: {e}")

    print(f"\nTable 1 shape: {metrics_df.shape} (methods × metrics)")
    print("\n" + "-" * 80)
    print("TABLE 1: Methods × Metrics")
    print("-" * 80)
    display(metrics_df)
else:
    print("ERROR: No curated metrics found!")

# =============================================================================
# TABLE 2: METHODS × PERFORMANCE
# =============================================================================
print("\n" + "=" * 60)
print("CREATING TABLE 2: Methods × Performance")
print("=" * 60)

perf_columns_list = [c for c in df.columns if c.startswith("perf_")]

if perf_columns_list:
    perf_df = df[perf_columns_list]

    perf_csv = PERF_DIR / "methods_x_performance.csv"
    perf_excel = PERF_DIR / "methods_x_performance.xlsx"

    perf_df.to_csv(perf_csv)
    print(f"✓ Saved CSV:   {perf_csv}")

    try:
        perf_df.to_excel(perf_excel, engine="openpyxl")
        print(f"✓ Saved Excel: {perf_excel}")
    except Exception as e:
        print(f"Note: Could not save Excel: {e}")

    print(f"\nTable 2 shape: {perf_df.shape} (methods × performance stats)")
    print(f"Performance columns: {perf_columns_list}")
    print("\nTABLE 2: Methods × Performance")
    display(perf_df)
else:
    print("No performance columns found")

# =============================================================================
# TABLE 3: CONFIG PARAMETERS
# =============================================================================
print("\n" + "=" * 60)
print("CREATING TABLE 3: Config Parameters")
print("=" * 60)

config_columns_list = [c for c in df.columns if c.startswith("cfg_")]

if config_columns_list:
    config_df = df[config_columns_list]

    config_csv = CONFIG_DIR / "methods_x_configs.csv"
    config_json = CONFIG_DIR / "methods_x_configs.json"
    config_excel = CONFIG_DIR / "methods_x_configs.xlsx"

    config_df.to_csv(config_csv)
    print(f"✓ Saved CSV:  {config_csv}")

    try:
        config_df.reset_index().to_json(config_json, orient="records", indent=2)
        print(f"✓ Saved JSON: {config_json}")
    except Exception as e:
        print(f"Note: Could not save JSON: {e}")

    try:
        config_df.to_excel(config_excel, engine="openpyxl")
        print(f"✓ Saved Excel: {config_excel}")
    except Exception as e:
        print(f"Note: Could not save Excel: {e}")

    print(f"\nTable 3 shape: {config_df.shape} (methods × config parameters)")
    print("\nTABLE 3: Config Parameters")
    display(config_df)
else:
    print("No config columns found")

# =============================================================================
# SAVE COMPLETE COMBINED DATAFRAME
# =============================================================================
print("\n" + "=" * 60)
print("SAVING COMPLETE COMBINED DATA")
print("=" * 60)

complete_csv = OUT_DIR / "complete_combined_data.csv"
complete_excel = OUT_DIR / "complete_combined_data.xlsx"

df.to_csv(complete_csv)
print(f"✓ Saved complete CSV:   {complete_csv}")

try:
    df.to_excel(complete_excel, engine="openpyxl")
    print(f"✓ Saved complete Excel: {complete_excel}")
except Exception as e:
    print(f"Note: Could not save complete Excel: {e}")

print(f"\nComplete data shape: {df.shape} (methods × all columns)")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total methods processed : {len(df)}")
print(f"Unique methods          : {df.index.nunique()}")
print(f"Methods                 : {sorted(df.index.unique())}")

print(f"\n✓ Table 1 (Metrics):     {METRICS_DIR}")
if available_metrics:
    print(f"  - {metrics_df.shape[0]} methods × {metrics_df.shape[1]} metrics")

print(f"\n✓ Table 2 (Performance): {PERF_DIR}")
if perf_columns_list:
    print(f"  - {perf_df.shape[0]} methods × {perf_df.shape[1]} performance stats")

print(f"\n✓ Table 3 (Configs):     {CONFIG_DIR}")
if config_columns_list:
    print(f"  - {config_df.shape[0]} methods × {config_df.shape[1]} config parameters")

print(f"\n✓ Complete Combined Data: {OUT_DIR}")
print(f"  - {df.shape[0]} methods × {df.shape[1]} total columns")

print("\n" + "=" * 60)
print("DONE!")
print("=" * 60)

Scanning runs directory: /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs
Expected methods: ['bbknn', 'combat', 'fastmnn', 'harmony', 'liger', 'scanorama', 'scanvi', 'scgen', 'scvi', 'seurat']


Successfully processed 10 methods

CREATING TABLE 1: Methods × Metrics

Found 15/15 curated metrics:
   1. NMI
   2. ARI
   3. cell_type_ASW
   4. isolated_label_F1
   5. isolated_label_ASW
   6. cell_cycle_conservation
   7. hvg_conservation
   8. trajectory_conservation
   9. trajectory_conservation_note
  10. batch_ASW
  11. pcr_comparison
  12. kBET
  13. graph_connectivity
  14. iLISI
  15. cLISI

✓ Saved CSV:   /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/metrics/methods_x_metrics.csv
✓ Saved Excel: /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/metrics/methods_x_metrics.xlsx

Table 1 shape: (10, 15) (methods × metrics)

---------------------------

,NMI,ARI,cell_type_ASW,isolated_label_F1,isolated_label_ASW,cell_cycle_conservation,hvg_conservation,trajectory_conservation,trajectory_conservation_note,batch_ASW,pcr_comparison,kBET,graph_connectivity,iLISI,cLISI
method,,,,,,,,,,,,,,,
bbknn,0.666655,0.504289,0.435823,0.239807,0.508508,0.416208,1.000000,None,compute_trajectory=False,0.788681,0.766331,0.298003,0.924987,0.414921,0.959893
combat,0.376499,0.092760,0.439617,0.209980,0.454527,0.685436,0.378444,None,compute_trajectory=False,0.718021,0.999709,0.087532,0.922861,0.023165,0.983317
fastmnn,0.557096,0.222330,0.468408,0.271428,0.469748,0.878926,1.000000,None,compute_trajectory=False,0.754622,0.061139,0.236184,0.977921,0.178234,0.986155
harmony,0.576121,0.258728,0.506665,0.253674,0.503188,0.937175,0.987111,None,compute_trajectory=False,0.856568,0.532701,0.241079,0.958437,0.179382,0.992001
liger,0.041572,0.003404,0.469871,0.012696,0.440340,0.047233,1.000000,None,compute_trajectory=False,0.918221,0.919194,0.044644,0.545266,0.260138,0.842415
scanorama,0.581131,0.211942,0.527313,0.412624,0.563808,0.862219,0.197778,None,compute_trajectory=False,0.828882,0.099069,0.208599,0.980123,0.096374,0.995771
scanvi,0.613120,0.317433,0.522927,0.275281,0.557214,0.589543,1.000000,None,compute_trajectory=False,0.735318,0.337907,0.266223,0.902585,0.203231,0.996229
scgen,0.632275,0.319534,0.544916,0.511116,0.569677,0.706642,1.000000,None,compute_trajectory=False,0.801752,0.563691,0.333005,0.979557,0.154911,0.997728
scvi,0.563884,0.231148,0.472264,0.290060,0.542251,0.687883,1.000000,None,NaN,0.790528,0.726687,0.177289,0.899853,0.105844,0.993007



CREATING TABLE 2: Methods × Performance
✓ Saved CSV:   /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/performance/methods_x_performance.csv
✓ Saved Excel: /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/performance/methods_x_performance.xlsx

Table 2 shape: (10, 4) (methods × performance stats)
Performance columns: ['perf_total_time_s', 'perf_peak_rss_gb', 'perf_peak_gpu_gb', 'perf_num_steps']

TABLE 2: Methods × Performance


,perf_total_time_s,perf_peak_rss_gb,perf_peak_gpu_gb,perf_num_steps
method,,,,
bbknn,33887.037006,12.587761,NaN,8
combat,3685.576866,18.974804,NaN,9
fastmnn,5145.471249,15.704380,NaN,9
harmony,4017.108272,13.061909,NaN,8
liger,3949.163900,12.250557,NaN,10
scanorama,20869.791608,53.546482,NaN,7
scanvi,4729.567897,14.421452,0.018281,9
scgen,4743.395213,13.796432,0.016176,12
scvi,3415.854830,14.169247,0.017756,10



CREATING TABLE 3: Config Parameters
✓ Saved CSV:  /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/configs/methods_x_configs.csv
✓ Saved JSON: /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/configs/methods_x_configs.json
✓ Saved Excel: /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/configs/methods_x_configs.xlsx

Table 3 shape: (10, 82) (methods × config parameters)

TABLE 3: Config Parameters


,cfg_batch_key,cfg_label_key,cfg_n_pcs,cfg_scale_before_pca,cfg_scale_max_value,cfg_pca_solver,cfg_neighbors_within_batch,cfg_trim,cfg_set_op_mix_ratio,cfg_local_connectivity,...,cfg_early_stopping_patience,cfg_num_workers,cfg_pin_memory,cfg_dropout_rate,cfg_enable_progress_bar,cfg_reference_batch,cfg_input_layer_counts,cfg_mode,cfg_k_anchor,cfg_dims
method,,,,,,,,,,,,,,,,,,,,,
bbknn,dataset,major_celltype_l1,50.0,False,10.0,arpack,5.0,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
combat,dataset,major_celltype_l1,50.0,False,10.0,arpack,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fastmnn,dataset,major_celltype_l1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
harmony,dataset,major_celltype_l1,50.0,NaN,NaN,arpack,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
liger,dataset,major_celltype_l1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scanorama,dataset,major_celltype_l1,50.0,False,10.0,arpack,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scanvi,dataset,major_celltype_l1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,4.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scgen,dataset,major_celltype_l1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,25.0,4.0,False,0.1,True,NaN,NaN,NaN,NaN,NaN
scvi,dataset,major_celltype_l1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,4.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN



SAVING COMPLETE COMBINED DATA
✓ Saved complete CSV:   /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/complete_combined_data.csv
✓ Saved complete Excel: /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/complete_combined_data.xlsx

Complete data shape: (10, 109) (methods × all columns)

SUMMARY
Total methods processed : 10
Unique methods          : 10
Methods                 : ['bbknn', 'combat', 'fastmnn', 'harmony', 'liger', 'scanorama', 'scanvi', 'scgen', 'scvi', 'seurat']

✓ Table 1 (Metrics):     /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/metrics
  - 10 methods × 15 metrics

✓ Table 2 (Performance): /data1/esraa/Thesis-Project/Results/Integration/Quantitive_metrics_scripts_and_results/Final_results_tables/performance
  - 10 methods × 4 performance stats

✓ Table 3 (Configs):     /data1/esraa/Thesis-P